In [1]:
print("hello world")

hello world


In [2]:
import pandas as pd

In [3]:
import pandas as pd
import os

# Now you can use just the filenames
books_df = pd.read_csv("03_Library Systembook.csv")
customers_df = pd.read_csv("03_Library SystemCustomers.csv")

print("Books loaded:", books_df.shape)
print("Customers loaded:", customers_df.shape)


Books loaded: (114, 6)
Customers loaded: (9, 2)


In [4]:
# ── Clean books_df ──────────────────────────────────────────────
# Remove rows where ALL columns are empty
books_df = books_df.dropna(how='all')

# Remove rows where ANY column is empty (use this if you want stricter cleaning)
# books_df = books_df.dropna(how='any')

# Replace blank strings/whitespace with NaN then drop them
books_df = books_df.replace(r'^\s*$', float('nan'), regex=True)
books_df = books_df.dropna(how='all')

# Strip whitespace from all text columns
books_df = books_df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)


# ── Check results ───────────────────────────────────────────────
print("Books missing values after cleaning:")
print(books_df.isnull().sum())
print("Books rows remaining:", len(books_df))

Books missing values after cleaning:
Id                        0
Books                     1
Book checkout             0
Book Returned             0
Days allowed to borrow    0
Customer ID               1
dtype: int64
Books rows remaining: 21


In [5]:
# ── Clean customers_df ──────────────────────────────────────────
customers_df = customers_df.dropna(how='all')

customers_df = customers_df.replace(r'^\s*$', float('nan'), regex=True)
customers_df = customers_df.dropna(how='all')

customers_df = customers_df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# ── Check results ───────────────────────────────────────────────
print("\nCustomers missing values after cleaning:")
print(customers_df.isnull().sum())
print("Customers rows remaining:", len(customers_df))


Customers missing values after cleaning:
Customer ID      0
Customer Name    0
dtype: int64
Customers rows remaining: 8


In [6]:
# ── Step 1: Remove all empty rows ──────────────────────────────
books_df = books_df.dropna(how='all')
print("Rows after removing empty rows:", len(books_df))

# ── Step 2: Remove row 21 (missing Book and Customer ID) ───────
books_df = books_df.dropna(subset=['Books', 'Customer ID'])
print("Rows after removing missing books/customers:", len(books_df))

# ── Step 3: Clean the dates - remove extra quotes ──────────────
books_df['Book checkout'] = books_df['Book checkout'].str.replace('"', '')
books_df['Book checkout'] = pd.to_datetime(books_df['Book checkout'], dayfirst=True, errors='coerce')
books_df['Book Returned'] = pd.to_datetime(books_df['Book Returned'], dayfirst=True, errors='coerce')

# ── Step 4: Remove duplicates ───────────────────────────────────
books_df = books_df.drop_duplicates()
print("Rows after removing duplicates:", len(books_df))

# ── Step 5: Flag the date typo (2063) for review ───────────────
print("\nSuspect dates (year not 2023):")
print(books_df[books_df['Book checkout'].dt.year != 2023][['Id','Books','Book checkout']])

# ── Preview cleaned data ────────────────────────────────────────
books_df.head(25)

Rows after removing empty rows: 21
Rows after removing missing books/customers: 20
Rows after removing duplicates: 20

Suspect dates (year not 2023):
      Id               Books Book checkout
6    7.0                  IT    2063-04-10
16  17.0  The Bloody Chamber           NaT


,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
0,1.0,Catcher in the Rye,2023-02-20,2023-02-25,2 weeks,1.0
1,2.0,Lord of the rings the two towers,2023-03-24,2023-03-21,2 weeks,2.0
2,3.0,Lord of the rings the return of the kind,2023-03-29,2023-03-25,2 weeks,3.0
3,4.0,The hobbit,2023-04-02,2023-03-25,2 weeks,4.0
4,5.0,Dune,2023-04-02,2023-03-25,2 weeks,5.0
5,6.0,Little Women,2023-04-02,2023-05-01,2 weeks,1.0
6,7.0,IT,2063-04-10,2023-04-03,2 weeks,6.0
7,8.0,Misery,2023-04-15,2023-04-03,2 weeks,7.0
8,9.0,Catch 22,2023-04-15,2023-04-16,2 weeks,7.0
9,10.0,Animal Farm,2023-04-20,2023-04-24,2 weeks,2.0


# Ask questions reffering to the unexplainable date for customer 17 and the wrong year on ID 7



In [7]:
# ── Step 1: Remove the NaN row ──────────────────────────────────
customers_df = customers_df.dropna(how='all')

# ── Step 2: Fix ID column from float to integer ─────────────────
customers_df['Customer ID'] = customers_df['Customer ID'].astype(int)

# ── Step 3: Check for missing customer IDs ──────────────────────
expected_ids = set(range(1, customers_df['Customer ID'].max() + 1))
actual_ids = set(customers_df['Customer ID'])
missing_ids = expected_ids - actual_ids

print("Missing Customer IDs:", missing_ids)

# ── Step 4: Preview cleaned data ────────────────────────────────
print("\nCleaned customers dataset:")
print(customers_df)

Missing Customer IDs: {4}

Cleaned customers dataset:
   Customer ID     Customer Name
0            1          Jane Doe
1            2        John Smith
2            3        Dan Reeves
4            5    William Holden
5            6     Jaztyn Forest
6            7     Jackie Irving
7            8  Matthew Stirling
8            9         Emory Ted


# Notice that there are two customers missing from this data set making it inaccurate

In [8]:
import os

# Get the folder where your notebook lives
notebook_folder = os.path.dirname(os.path.abspath('__file__'))

# Save to the same folder as your notebook
books_df.to_csv(os.path.join(notebook_folder, 'cleaned_books.csv'), index=False)
customers_df.to_csv(os.path.join(notebook_folder, 'cleaned_customers.csv'), index=False)

print("Saved to:", notebook_folder)

Saved to: c:\Users\Admin\Desktop\2026-05-DE5M5


In [9]:
# ── Save cleaned books dataset ──────────────────────────────────
books_df.to_csv('cleaned_books.csv', index=False)
print("Books saved successfully!")

# ── Save cleaned customers dataset ─────────────────────────────
customers_df.to_csv('cleaned_customers.csv', index=False)
print("Customers saved successfully!")

Books saved successfully!
Customers saved successfully!


In [10]:


# ── Step 1: Identify all errors in books_df ─────────────────────

# Wrong year (2063)
wrong_year = books_df[books_df['Book checkout'].dt.year != 2023].copy()
wrong_year['Error Reason'] = 'Invalid checkout year (2063)'

# Duplicate rows
duplicate_rows = books_df[books_df.duplicated(keep='first')].copy()
duplicate_rows['Error Reason'] = 'Duplicate row'

# Missing Book or Customer ID
missing_books = books_df[books_df['Books'].isna() | books_df['Customer ID'].isna()].copy()
missing_books['Error Reason'] = 'Missing Book or Customer ID'

# Customers not in customers_df
valid_customer_ids = customers_df['Customer ID'].tolist()
missing_customers = books_df[~books_df['Customer ID'].isin(valid_customer_ids)].copy()
missing_customers['Error Reason'] = 'Customer ID not found in customers dataset'

# ── Step 2: Combine all errors into one errors dataset ──────────
errors_df = pd.concat([wrong_year, duplicate_rows, missing_books, missing_customers])
errors_df = errors_df.drop_duplicates()

print("=== ERRORS FOUND ===")
print(errors_df[['Id', 'Books', 'Book checkout', 'Customer ID', 'Error Reason']])
print(f"\nTotal errors: {len(errors_df)}")

# ── Step 3: Remove errors from books_df ─────────────────────────
cleaned_books_df = books_df[~books_df['Id'].isin(errors_df['Id'])].copy()
print(f"\nRows in cleaned dataset: {len(cleaned_books_df)}")
print(f"Rows removed as errors: {len(books_df) - len(cleaned_books_df)}")

# ── Step 4: Clean customers_df ──────────────────────────────────
# Flag missing customer IDs
missing_customer_ids = customers_df[customers_df['Customer ID'].isna() | 
                                     customers_df['Customer Name'].isna()].copy()
missing_customer_ids['Error Reason'] = 'Missing Customer ID or Name'

errors_customers_df = missing_customer_ids.copy()

# Remove errors from customers
cleaned_customers_df = customers_df.dropna(subset=['Customer ID', 'Customer Name'])
cleaned_customers_df['Customer ID'] = cleaned_customers_df['Customer ID'].astype(int)

print("\n=== CUSTOMER ERRORS ===")
print(errors_customers_df)

# ── Step 5: Save all four datasets ──────────────────────────────
cleaned_books_df.to_csv('cleaned_books.csv', index=False)
cleaned_customers_df.to_csv('cleaned_customers.csv', index=False)
errors_df.to_csv('errors_books.csv', index=False)
errors_customers_df.to_csv('errors_customers.csv', index=False)

print("\n=== ALL FILES SAVED ===")
print("✓ cleaned_books.csv")
print("✓ cleaned_customers.csv")
print("✓ errors_books.csv")
print("✓ errors_customers.csv")

=== ERRORS FOUND ===
      Id               Books Book checkout  Customer ID  \
6    7.0                  IT    2063-04-10          6.0   
16  17.0  The Bloody Chamber           NaT          3.0   
3    4.0          The hobbit    2023-04-02          4.0   
18  19.0             Dracula    2023-06-10         10.0   

                                  Error Reason  
6                 Invalid checkout year (2063)  
16                Invalid checkout year (2063)  
3   Customer ID not found in customers dataset  
18  Customer ID not found in customers dataset  

Total errors: 4

Rows in cleaned dataset: 16
Rows removed as errors: 4

=== CUSTOMER ERRORS ===
Empty DataFrame
Columns: [Customer ID, Customer Name, Error Reason]
Index: []

=== ALL FILES SAVED ===
✓ cleaned_books.csv
✓ cleaned_customers.csv
✓ errors_books.csv
✓ errors_customers.csv


# Data engiering metrics
- Understanding how many errors were tracked and where from
- Where is the customer data being connected to for updates
- Total numbers that were cleaned
- How many books has one person got

